<a href="https://colab.research.google.com/github/anokhina-rgb/Google-Colabs/blob/main/Copy_of_Accounting_Book_Generator_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    from docx import Document
    from docx.shared import Pt, Inches
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from googletrans import Translator
except ImportError:
    !pip install python-docx googletrans==4.0.0-rc1
    from docx import Document
    from docx.shared import Pt, Inches
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    from googletrans import Translator

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import io
import base64
import re

translator = Translator()

def get_translation(text, dest='uk'):
    try:
        return translator.translate(text[:5000], dest=dest).text
    except:
        return "[Помилка перекладу]"

def clean_text(text):
    # Видаляємо все технічне "сміття"
    text = re.sub(r'(e-ISSN|https?://|Received:|Accepted:|Available online:|Volume|Article Info|DOI|Highlights|Journal Web Page).*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\[\d+(?:-\d+)?(?:,\s*\d+)*\]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def extract_multi_terms(text):
    # Терміни: 2-3 слова з великої літери
    terms = re.findall(r'\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+){1,2})\b', text)
    unique_terms = list(dict.fromkeys(terms))[:15]
    return {term: get_translation(term) for term in unique_terms}

def create_book(text):
    text = clean_text(text)
    sentences = re.split(r'(?<=[.!?])\s+', text)
    doc = Document()

    # Титульна сторінка
    doc.add_heading('Навчальна книга: Сучасні бухгалтерські системи', 0)
    doc.add_page_break()

    chunk_size = 20
    chapter_count = 0

    for i in range(0, len(sentences), chunk_size):
        chunk = sentences[i:i + chunk_size]
        if len(chunk) < 20: continue

        chapter_text = " ".join(chunk)
        terms = extract_multi_terms(chapter_text)

        # Заголовок розділу
        doc.add_heading(f"Розділ {chapter_count + 1}: {list(terms.keys())[0] if terms else 'Аналіз'}", level=1)

        # Основний текст
        p = doc.add_paragraph(chapter_text)
        p.paragraph_format.line_spacing = 1.5

        # Глосарій
        doc.add_heading('Глосарій термінів', level=2)
        table = doc.add_table(rows=1, cols=2)
        table.style = 'Table Grid'
        hdr_cells = table.rows[0].cells
        hdr_cells[0].text = 'Термін'
        hdr_cells[1].text = 'Переклад'

        for eng, ukr in terms.items():
            row_cells = table.add_row().cells
            row_cells[0].text = eng
            row_cells[1].text = ukr

        # Переклад
        doc.add_heading('Переклад розділу', level=2)
        doc.add_paragraph(get_translation(chapter_text))

        doc.add_page_break()
        chapter_count += 1
        if chapter_count >= 4: break

    bio = io.BytesIO()
    doc.save(bio)
    return bio.getvalue()

uploader = widgets.FileUpload(accept='.txt', multiple=False)
btn = widgets.Button(description="Створити книгу", layout=widgets.Layout(width='200px', height='40px'))
output = widgets.Output()

def on_click(b):
    output.clear_output()
    with output:
        if not uploader.value: return print("Завантажте .txt")
        val = uploader.value
        file_info = val[0] if isinstance(val, (list, tuple)) else next(iter(val.values()))
        content = file_info['content'].decode('utf-8')
        docx_data = create_book(content)
        b64 = base64.b64encode(docx_data).decode()
        display(HTML(f'<a href="data:application/vnd.openxmlformats-officedocument.wordprocessingml.document;base64,{b64}" download="Accounting_Book.docx">📥 Завантажити книгу</a>'))

btn.on_click(on_click)
display(widgets.VBox([uploader, btn, output]))

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 5.4 MB/s eta 0:00:00
  Created wheel for googletrans: filename=googletrans-4.0.0rc1-py3-none-any.whl size=17396 sha256=69a54a42eeebf2d1f97f2fbb75524f48102418d58799ae8f621fdd1c07079d8c
  Stored in directory: /root/.cache/pip/wheels/95/0f/04/b17a72024b56a60e499ce1a6313d283ed5ba332407155bee03
Successfully built googletrans
  Attempting uninstall: hyperf